In [ ]:
from pathlib import Path
# 
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import matplotlib.pyplot as plt # ploting graphs and chart 
from datetime import timedelta # for time formatting
import time
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk(f"{Path(__file__).parent.parent}/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import cudf
cudf.set_option("copy_on_write", True)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

df  = pd.read_csv(f"{Path(__file__).parent.parent}/input/imdb-official-movies-dataset/title-ratings.csv")
df2 = pd.read_csv(f"{Path(__file__).parent.parent}/input/imdb-official-movies-dataset/title-metadata.csv")
factor = 0.1

# Merging Rating and MetaData datsets into single dataframe w.r.t common column
main_df = pd.merge(df,df2,on='tconst')
main_df = main_df.sample(frac=factor, random_state=0)
main_df.head()

In [ ]:
### cell 1 ###

# creating a copy 
imdb_df = main_df.copy()

In [ ]:
### cell 2 ###

imdb_df['runtimeMinutes'].value_counts()

In [ ]:
### cell 3 ###

imdb_df['runtime_delta'] = pd.to_numeric(imdb_df['runtimeMinutes'], errors='coerce', downcast="integer")

In [ ]:
### cell 4 ###

# Using 'pd.to_timedelta' to convert minutes into desired format
imdb_df['runtime_delta'] = pd.to_timedelta(imdb_df['runtime_delta'], unit='m')

In [ ]:
### cell 5 ###

imdb_df['isAdult'].unique()

In [ ]:
### cell 6 ###

# Replace string and int by boolean
# we will leave out '2014' and '2020' as it is as there`s no way to confirm If they Adult or not not
imdb_df['isAdult'] = imdb_df['isAdult'].map({'0': 'Non_Adult_Title', '1': 'Adult_Title'})

In [ ]:
### cell 7 ###

imdb_df.isAdult = imdb_df.isAdult.fillna('Unrated')
imdb_df['isAdult'].unique()

In [ ]:
### cell 8 ###

# Now joinning them
# STEFANOS-DISABLE-FOR-MODIN
### ORIGINAL ###
# imdb_df['premiere_timeline'] = imdb_df[['startYear','endYear']].stack().groupby(level=0).agg('-'.join)
### COMPATIBLE WITH MODIN ###
# I tried a bunch of things, here's one:
#   imdb_df['premiere_timeline'] = imdb_df[['startYear','endYear']].stack().groupby(level=0).apply('-'.join)
# They don't seem to work. The problem seems to be with groupby(level=0). Generally, I'm not sure that Modin can
# groupby MultiIndex's.
# So, below I do the computation up to the stacking, just so that we can keep as much of it as possible (in case
# Modin, or some other system that uses the same benchmarks, can optimize it; we don't want to deprive this
# opportunity). Unfortunately, we cannot assign the result of stacking to premiere_timeline as they don't match.
# I just assign a dumb value to premiere_timeline so that it exists in the DF. It's not used later.

_ = imdb_df[['startYear','endYear']].stack()
imdb_df['premiere_timeline'] = 0

In [ ]:
### cell 9 ###

# Drop unneccessory columns from dataframe
columns_to_keep = ['tconst','averageRating','numVotes', 'primaryTitle', 'titleType',
                   'isAdult','premiere_timeline','runtime_delta']

imdb_df = imdb_df[columns_to_keep]

# Also renaming them with approriate name
renamed_cols = {'tconst':'IMDB_ID','averageRating':'Avg_Rating','numVotes':'Total_Votes',
                'titleType':'Title_Category', 'primaryTitle':'Title_Name', 'isAdult':'IN-18+'
                ,'premiere_timeline':'Air_time','runtime_delta':'Title_Runtime_Length'}

imdb_df.rename(columns = renamed_cols,inplace=True)
imdb_df.head()

In [ ]:
### cell 10 ###

top_10_highest = imdb_df.groupby(['Title_Name'])[
    ['Avg_Rating','Total_Votes']].max().sort_values(by=['Avg_Rating','Total_Votes'
                                                       ], ascending = False).reset_index().head(10)

top_10_highest

In [ ]:
### cell 11 ###

highest_rated_type = imdb_df.groupby(['Title_Category'])[['Avg_Rating','Total_Votes']].mean(
).sort_values(by=['Avg_Rating','Total_Votes'], ascending = False).reset_index().round(decimals = 1)

highest_rated_type

In [ ]:
### cell 12 ###

longest_runtime = imdb_df.loc[imdb_df.groupby(['Title_Category'], 
                                              sort=False)['Title_Runtime_Length'].idxmax()
                             ][['Title_Category', 'Title_Name', 'Title_Runtime_Length']]

longest_runtime.sort_values(by='Title_Runtime_Length', ascending = False).reset_index(drop=True)

In [ ]:
### cell 13 ###

shortest_runtime = imdb_df.loc[imdb_df.groupby(['Title_Category'], 
                                              sort=False)['Title_Runtime_Length'].idxmin()
                             ][['Title_Category', 'Title_Name', 'Title_Runtime_Length']]

shortest_runtime.reset_index(drop=True)

In [ ]:
### cell 14 ###

wanna_adult_or_not = imdb_df.groupby(['Title_Category','IN-18+']
                                    )['Avg_Rating'].agg('mean').round(1).unstack(fill_value= 0) 
wanna_adult_or_not.reset_index(inplace = True)
##### Visualise the chart in stack manner of bar type

# STEFANOS: Disable plotting
# plt.rcParams['figure.figsize'] = [15, 8]
# wanna_adult_or_not.plot(x='Title_Category', kind='bar', stacked=True, 
#                         title='Average Ratings of Non_Adult Rated Titles vs Adult Rated Titles')
# plt.xlabel('Title Categories')
# plt.ylabel('Average Ratings')
# plt.show()

In [ ]:
### cell 15 ###

imdb_df.groupby(['Title_Category','IN-18+']
                                    )['Avg_Rating'].agg('mean').round(1).unstack(fill_value= 0)